# Week 4 Monday Exercise 2 — Synthetic Identity Decision Tree

This notebook shows how a simple tree can separate low-risk from higher-risk synthetic identity cases.


In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text


In [2]:
df = pd.read_csv('week4_monday_synthetic_identity_dataset.csv')
df


,record_id,customer_name,customer_type,file_depth,account_age,identity_consistency,phone_reuse,email_reuse,address_reuse,device_reuse,ip_risk,synthetic_indicator,manual_risk_label,feature_score,risk_tier,recommended_action
0,S001,North Star Retail,Retail,Stable,1-12 mo,Consistent,Same,Same,Same,Same,Low,No,Low,8,High,Investigate and hold for review
1,S002,Riverline App,New,Thin-file,<1 mo,Mismatched,Reused,Reused,Reused,Shared,High,Yes,High,10,High,Investigate and hold for review
2,S003,Cedar Credit,New,Thin-file,<1 mo,Partial,Shared,Same,Same,Shared,Medium,Yes,High,8,High,Investigate and hold for review
3,S004,Metro Supplies,Business,Moderate,12+ mo,Consistent,Same,Same,Same,Same,Low,No,Low,1,Low,Standard onboarding
4,S005,Echo Finance,New,Thin-file,<1 mo,Mismatched,Shared,Shared,Reused,Shared,High,Yes,High,9,High,Investigate and hold for review
5,S006,Pine Local,Retail,Stable,1-12 mo,Partial,Same,Same,Same,Same,Low,No,Low,2,Low,Standard onboarding
6,S007,Summit Pay,New,Thin-file,<1 mo,Partial,Shared,Reused,Same,Shared,High,Yes,High,8,High,Investigate and hold for review
7,S008,Harbor Works,Business,Moderate,12+ mo,Consistent,Same,Shared,Same,Same,Low,No,Low,1,Low,Standard onboarding


## Target label
Use the manual risk tier as the target.


In [3]:
y = df['risk_tier'].map({'Low':0,'Medium':1,'High':1})
X = pd.get_dummies(df[['customer_type','file_depth','account_age','identity_consistency','phone_reuse','email_reuse','address_reuse','device_reuse','ip_risk','synthetic_indicator']], drop_first=False)
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X, y)
df['predicted'] = clf.predict(X)
df[['record_id','customer_name','risk_tier','predicted']]


,record_id,customer_name,risk_tier,predicted
0,S001,North Star Retail,High,1
1,S002,Riverline App,High,1
2,S003,Cedar Credit,High,1
3,S004,Metro Supplies,Low,0
4,S005,Echo Finance,High,1
5,S006,Pine Local,Low,0
6,S007,Summit Pay,High,1
7,S008,Harbor Works,Low,0


## Decision rules


In [4]:
print(export_text(clf, feature_names=list(X.columns)))


|--- phone_reuse_Same <= 0.50
|   |--- class: 1
|--- phone_reuse_Same >  0.50
|   |--- account_age_1-12 mo <= 0.50
|   |   |--- class: 0
|   |--- account_age_1-12 mo >  0.50
|   |   |--- identity_consistency_Partial <= 0.50
|   |   |   |--- class: 1
|   |   |--- identity_consistency_Partial >  0.50
|   |   |   |--- class: 0

